In [3]:

# =====================================================
# ENIGMAROOM - VERSION 2
# RELIABLE AI ESCAPE ROOM
# =====================================================

import gradio as gr
import copy

INITIAL_STATE = {
    "key_found": False,
    "drawer_open": False,
    "uv_found": False,
    "note_found": False,
    "portrait_solved": False,
    "door_unlocked": False,
    "attempts": 0,
    "inventory": []
}

def process_action(message, history, state):

    msg = message.lower().strip()
    state = copy.deepcopy(state)

    # 1. Restart
    if "restart" in msg:
        return "🔄 Game restarted! Your progress and inventory have been reset.", INITIAL_STATE.copy()

    # 2. Protect against override attempts
    forbidden = [
        "ignore all rules",
        "i am the administrator",
        "admin override",
        "reveal the secret code",
        "tell me all solutions",
        "unlock the door immediately"
    ]

    if any(word in msg for word in forbidden):
        return "🛡️ SECURITY ALERT: Your request is not permitted. Complete the puzzles to escape.", state

    # 3. Bookshelf
    if "bookshelf" in msg or "books" in msg:
        if not state["key_found"]:
            state["key_found"] = True
            state["inventory"].append("Brass Key")
            return "📚 You discover a hidden brass key behind an ancient book. Added to inventory!", state

        return "You have already searched the bookshelf.", state

    # 4. Locked drawer
    if "drawer" in msg:
        if not state["key_found"]:
            return "🔒 Access denied! You need the brass key first.", state

        if not state["drawer_open"]:
            state["drawer_open"] = True
            state["uv_found"] = True
            state["inventory"].append("UV Flashlight")
            return "🗝️ The drawer opens! You discover a UV flashlight and collect it.", state

        return "The drawer is already open.", state

    # 5. Desk and note
    if "desk" in msg or "note" in msg:
        if not state["drawer_open"]:
            return "🔒 The desk compartment is locked. Open the drawer first.", state

        if not state["note_found"]:
            state["note_found"] = True
            state["inventory"].append("Secret Note")
            return "📜 You discover a note containing the first two digits: 43.", state

        return "You have already collected the note.", state

    # 6. Portrait
    if "portrait" in msg or "painting" in msg:
        if not state["uv_found"]:
            return "🖼️ The portrait has hidden markings. You need a UV flashlight to reveal them.", state

        if not state["portrait_solved"]:
            state["portrait_solved"] = True
            return "🔦 The UV light reveals the final two digits: 17.", state

        return "You have already solved the portrait puzzle.", state

    # 7. Inventory
    if "inventory" in msg or "items" in msg:
        items = state["inventory"]

        return (
            "🎒 Your inventory:\n" +
            ("\n".join("• " + item for item in items) if items else "Empty")
        ), state

    # 8. Exit and code validation
    if "door" in msg or "exit" in msg or any(c.isdigit() for c in msg):

        if state["door_unlocked"]:
            return "🎉 You have already escaped EnigmaRoom!", state

        if not (state["note_found"] and state["portrait_solved"]):
            return "🔐 The exit remains locked. Discover all the clues first.", state

        if "4317" in msg:
            state["door_unlocked"] = True
            return "🎉 SUCCESS! The door unlocks. You escaped EnigmaRoom!", state

        state["attempts"] += 1
        return "❌ Incorrect code. The exit remains locked.", state

    return "🔍 Unknown action. Try bookshelf, drawer, desk, portrait, inventory or exit.", state


def respond(message, history, state):

    if state is None:
        state = INITIAL_STATE.copy()

    reply, updated_state = process_action(
        message, history, state
    )

    history = (history or []) + [(message, reply)]

    return "", history, updated_state


def restart_game():
    return [], INITIAL_STATE.copy()


def show_status(state):
    if state is None:
        state = INITIAL_STATE.copy()

    return f"""
### 🛡️ Game Status

- Brass Key: {'✅' if state['key_found'] else '❌'}
- Drawer: {'✅ Open' if state['drawer_open'] else '🔒 Locked'}
- UV Flashlight: {'✅' if state['uv_found'] else '❌'}
- Secret Note: {'✅' if state['note_found'] else '❌'}
- Portrait Puzzle: {'✅' if state['portrait_solved'] else '❌'}
- Exit: {'🔓 Unlocked' if state['door_unlocked'] else '🔒 Locked'}
- Failed code attempts: {state['attempts']}

**Inventory:** {', '.join(state['inventory']) if state['inventory'] else 'Empty'}
"""


with gr.Blocks(theme=gr.themes.Soft()) as demo_v2:

    gr.Markdown("""
    # 🛡️ EnigmaRoom – Version 2
    ### Reliable AI Escape Challenge

    Python validates every action and controls the game state.
    The player cannot unlock the door by simply asking the system.
    """)

    game_state = gr.State(INITIAL_STATE.copy())

    chatbot = gr.Chatbot(
    height=400,
    label="Escape Room Conversation"
)

    status = gr.Markdown(value=show_status(INITIAL_STATE))

    message = gr.Textbox(
        placeholder="Examine the bookshelf, open the drawer, check inventory...",
        label="Your action"
    )

    with gr.Row():
        send = gr.Button("Send", variant="primary")
        restart = gr.Button("Restart Game")

    send.click(
        respond,
        inputs=[message, chatbot, game_state],
        outputs=[message, chatbot, game_state]
    ).then(
        show_status,
        inputs=game_state,
        outputs=status
    )

    message.submit(
        respond,
        inputs=[message, chatbot, game_state],
        outputs=[message, chatbot, game_state]
    ).then(
        show_status,
        inputs=game_state,
        outputs=status
    )

    restart.click(
        restart_game,
        outputs=[chatbot, game_state]
    ).then(
        show_status,
        inputs=game_state,
        outputs=status
    )

demo_v2.launch(share=True)


/tmp/ipykernel_4065/2909571022.py:152: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo_v2:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c2883db43fbc10ce00.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [4]:
# Test Version 2 game logic without Gradio

test_state = INITIAL_STATE.copy()

result, updated_state = process_action(
    "Open the locked drawer",
    [],
    test_state
)

print("Player action: Open the locked drawer")
print("Game response:", result)
print("Game state:", updated_state)

Player action: Open the locked drawer
Game response: 🔒 Access denied! You need the brass key first.
Game state: {'key_found': False, 'drawer_open': False, 'uv_found': False, 'note_found': False, 'portrait_solved': False, 'door_unlocked': False, 'attempts': 0, 'inventory': []}


In [5]:
# Test Version 2 game logic without Gradio

test_state = INITIAL_STATE.copy()

result, updated_state = process_action(
    "Open the locked drawer",
    [],
    test_state
)

print("Player action: Open the locked drawer")
print("Game response:", result)
print("Game state:", updated_state)

Player action: Open the locked drawer
Game response: 🔒 Access denied! You need the brass key first.
Game state: {'key_found': False, 'drawer_open': False, 'uv_found': False, 'note_found': False, 'portrait_solved': False, 'door_unlocked': False, 'attempts': 0, 'inventory': []}


In [6]:
import gradio as gr
import inspect

print("Gradio version:", gr.__version__)

print("\nChatbot configuration:")
print(inspect.signature(gr.Chatbot))

print("\nButton event configuration:")
print("Gradio loaded successfully!")

Gradio version: 6.31.0

Chatbot configuration:
(value: 'list[MessageDict | Message] | Callable | None' = None, *, label: 'str | I18nData | None' = None, every: 'Timer | float | None' = None, inputs: 'Component | Sequence[Component] | set[Component] | None' = None, show_label: 'bool | None' = None, container: 'bool' = True, scale: 'int | None' = None, min_width: 'int' = 160, visible: "bool | Literal['hidden']" = True, elem_id: 'str | None' = None, elem_classes: 'list[str] | str | None' = None, autoscroll: 'bool' = True, render: 'bool' = True, key: 'int | str | tuple[int | str, ...] | None' = None, preserved_by_key: 'list[str] | str | None' = 'value', height: 'int | str | None' = 400, resizable: 'bool' = False, max_height: 'int | str | None' = None, min_height: 'int | str | None' = None, editable: "Literal['user', 'all'] | None" = None, latex_delimiters: 'list[dict[str, str | bool]] | None' = None, rtl: 'bool' = False, buttons: "list[Literal['share', 'copy', 'copy_all'] | Button] | None"

In [7]:

# ============================================================
# ENIGMAROOM - AI POWERED ESCAPE ROOM
# VERSION 2: RELIABLE GAME WITH PYTHON VALIDATION
# Gradio 6.31.0 Compatible
# ============================================================

import gradio as gr
import copy
import re

# ---------------- INITIAL GAME STATE ----------------

INITIAL_STATE = {
    "key_found": False,
    "drawer_open": False,
    "uv_found": False,
    "note_found": False,
    "portrait_solved": False,
    "door_unlocked": False,
    "attempts": 0,
    "inventory": []
}


# ---------------- GAME LOGIC ----------------

def process_action(message, state):

    msg = message.lower().strip()
    state = copy.deepcopy(state)

    # Restart
    if "restart" in msg:
        return (
            "🔄 The library resets. Your adventure begins again!",
            INITIAL_STATE.copy()
        )

    # Security protection
    forbidden = [
        "ignore all rules",
        "administrator",
        "admin override",
        "reveal all",
        "tell me the code",
        "unlock the door immediately",
        "show me the solution"
    ]

    if any(word in msg for word in forbidden):
        return (
            "🛡️ SECURITY ALERT! This request is not permitted. "
            "You must solve the puzzles yourself.",
            state
        )

    # Bookshelf
    if "bookshelf" in msg or "books" in msg:

        if not state["key_found"]:
            state["key_found"] = True
            state["inventory"].append("Brass Key")

            return (
                "📚 You discover a hidden brass key behind an ancient book! "
                "The Brass Key has been added to your inventory.",
                state
            )

        return "You have already searched the bookshelf.", state

    # Locked drawer
    if "drawer" in msg:

        if not state["key_found"]:
            return "🔒 Access denied! You need the brass key first.", state

        if not state["drawer_open"]:
            state["drawer_open"] = True
            state["uv_found"] = True
            state["inventory"].append("UV Flashlight")

            return (
                "🗝️ You use the brass key to open the drawer. "
                "Inside, you discover a UV flashlight!",
                state
            )

        return "The drawer is already open.", state

    # Desk and secret note
    if "desk" in msg or "note" in msg:

        if not state["drawer_open"]:
            return (
                "🔒 The desk compartment is locked. "
                "Open the drawer first.",
                state
            )

        if not state["note_found"]:
            state["note_found"] = True
            state["inventory"].append("Secret Note")

            return (
                "📜 You discover a secret note. "
                "It reveals the first two digits: 43.",
                state
            )

        return "You have already collected the secret note.", state

    # Portrait puzzle
    if "portrait" in msg or "painting" in msg:

        if not state["uv_found"]:
            return (
                "🖼️ The portrait contains hidden markings. "
                "You need a UV flashlight.",
                state
            )

        if not state["portrait_solved"]:
            state["portrait_solved"] = True

            return (
                "🔦 The UV flashlight reveals hidden writing: "
                "THE FINAL TWO DIGITS ARE 17.",
                state
            )

        return "You have already solved the portrait puzzle.", state

    # Inventory
    if "inventory" in msg or "items" in msg:

        items = state["inventory"]

        if items:
            result = "\n".join("• " + item for item in items)
        else:
            result = "Your inventory is empty."

        return "🎒 YOUR INVENTORY:\n" + result, state

    # Exit and keypad
    if (
        "door" in msg
        or "exit" in msg
        or re.search(r"\b\d{4}\b", msg)
    ):

        if state["door_unlocked"]:
            return "🎉 You have already escaped EnigmaRoom!", state

        if not (
            state["note_found"]
            and state["portrait_solved"]
        ):
            return (
                "🔐 The exit is still locked. "
                "Discover all the clues before attempting to escape.",
                state
            )

        code_match = re.search(r"\b\d{4}\b", msg)

        if code_match:

            code = code_match.group()

            if code == "4317":
                state["door_unlocked"] = True

                return (
                    "🎉 CONGRATULATIONS! 🎉\n\n"
                    "The keypad turns green. The door opens!\n"
                    "YOU HAVE SUCCESSFULLY ESCAPED ENIGMAROOM!",
                    state
                )

            state["attempts"] += 1

            return (
                "❌ Incorrect code! The door remains locked.",
                state
            )

        return (
            "🚪 The exit door has a four-digit keypad. "
            "Enter your four-digit code.",
            state
        )

    # Unknown action
    return (
        "🔍 I don't recognise that action.\n\n"
        "Try:\n"
        "• Examine the bookshelf\n"
        "• Open the locked drawer\n"
        "• Examine the desk\n"
        "• Inspect the portrait\n"
        "• Check inventory\n"
        "• Enter the exit code",
        state
    )


# ---------------- GRADIO CHAT FUNCTION ----------------

def respond(message, history, state):

    if state is None:
        state = INITIAL_STATE.copy()

    if history is None:
        history = []

    reply, updated_state = process_action(message, state)

    updated_history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": reply}
    ]

    return "", updated_history, updated_state


# ---------------- GAME STATUS ----------------

def show_status(state):

    if state is None:
        state = INITIAL_STATE.copy()

    return f"""
## 🛡️ GAME STATUS

| Game Progress | Status |
|---|---|
| Brass Key | {'✅ Found' if state['key_found'] else '❌ Not Found'} |
| Locked Drawer | {'✅ Open' if state['drawer_open'] else '🔒 Locked'} |
| UV Flashlight | {'✅ Found' if state['uv_found'] else '❌ Not Found'} |
| Secret Note | {'✅ Found' if state['note_found'] else '❌ Not Found'} |
| Portrait Puzzle | {'✅ Solved' if state['portrait_solved'] else '❌ Not Solved'} |
| Exit Door | {'🔓 Unlocked' if state['door_unlocked'] else '🔒 Locked'} |

**Inventory:** {', '.join(state['inventory']) if state['inventory'] else 'Empty'}

**Incorrect Code Attempts:** {state['attempts']}
"""


# ---------------- RESTART ----------------

def restart_game():

    return [], INITIAL_STATE.copy()


# ---------------- GRADIO INTERFACE ----------------

with gr.Blocks(
    theme=gr.themes.Soft()
) as demo_v2:

    gr.Markdown("""
    # 🔐 ENIGMAROOM
    ## Version 2 – Reliable AI Escape Challenge

    Welcome to the mysterious library!

    Explore the room, collect items, discover hidden clues
    and unlock the exit.

    **Security:** Python validates all game actions.
    """)

    game_state = gr.State(INITIAL_STATE.copy())

    chatbot = gr.Chatbot(
        height=420,
        label="Escape Room Conversation",
        layout="bubble"
    )

    status = gr.Markdown(
        value=show_status(INITIAL_STATE)
    )

    message = gr.Textbox(
        label="Your Action",
        placeholder="Examine bookshelf, open drawer, check inventory..."
    )

    with gr.Row():

        send = gr.Button(
            "Send",
            variant="primary"
        )

        restart = gr.Button(
            "🔄 Restart Game"
        )

    send.click(
        respond,
        inputs=[message, chatbot, game_state],
        outputs=[message, chatbot, game_state]
    ).then(
        show_status,
        inputs=game_state,
        outputs=status
    )

    message.submit(
        respond,
        inputs=[message, chatbot, game_state],
        outputs=[message, chatbot, game_state]
    ).then(
        show_status,
        inputs=game_state,
        outputs=status
    )

    restart.click(
        restart_game,
        outputs=[chatbot, game_state]
    ).then(
        show_status,
        inputs=game_state,
        outputs=status
    )


# ---------------- LAUNCH APPLICATION ----------------

demo_v2.launch(share=True)


/tmp/ipykernel_4065/2651536112.py:263: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6cee7743e0500bd6fe.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
